# 🧠 CogniLink Demo

**Document-to-graph knowledge mapper powered by LangChain.**

This notebook:
1. Installs CogniLink from GitHub
2. Uploads a PDF (or uses a sample)
3. Builds a knowledge graph with LLM summaries
4. Displays the interactive visualization

## 1. Install CogniLink from GitHub

In [ ]:
!pip install -q git+https://github.com/akhilkanugolu/cognilink.git
!pip install -q langchain-openai PyPDF2

## 2. Set your OpenAI API Key

In [ ]:
import os
from getpass import getpass

# Securely input your API key (won't be displayed)
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Upload your PDF (or use a sample)

In [ ]:
from google.colab import files
from pathlib import Path

# Option A: Upload a PDF
print("Upload a PDF file (or skip to use a sample):")
try:
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"\n✅ Using uploaded file: {pdf_path}")
except Exception:
    # Option B: Use a sample text file
    print("\n⏭️ No file uploaded. Creating a sample document...")
    pdf_path = "sample_spec.txt"
    Path(pdf_path).write_text("""
# Authentication System Specification

## Overview
The system uses JWT-based authentication with refresh tokens.
Access tokens expire after 15 minutes. Refresh tokens expire after 7 days.

## Login Flow
1. User submits email + password
2. Server validates credentials against bcrypt hash
3. Server issues access_token (15min) and refresh_token (7d)
4. Client stores tokens in httpOnly cookies

## Security Requirements
- All passwords must be hashed with bcrypt (cost factor 12)
- Rate limiting: max 5 failed attempts per IP per 15 minutes
- Refresh tokens are single-use (rotation on each refresh)
- CSRF protection via double-submit cookie pattern
""")
    print(f"✅ Created sample: {pdf_path}")

## 4. Run CogniLink

In [ ]:
import cognilink
from langchain_openai import ChatOpenAI

# Create the model
model = ChatOpenAI(model="gpt-4o-mini")  # Uses OPENAI_API_KEY from env

# Run CogniLink
print("🚀 Building knowledge graph...\n")
result = cognilink.map(
    pdf_path,
    model=model,
    output_dir="./output"
)

print("✅ Done!\n")
print(f"📊 Nodes: {len(result['nodes'])}")
print(f"🔗 Edges: {len(result['edges'])}")
print(f"🌐 HTML: {result['html_path']}")
print(f"📋 JSON: {result['json_path']}")

## 5. View Node Summaries

In [ ]:
for node in result["nodes"]:
    print(f"\n🔹 {node['id']} [{node['state']}]")
    print(f"   Type: {node['source_type']}")
    print(f"   Summary: {node['summary']}")

## 6. Display Interactive Graph

In [ ]:
from IPython.display import HTML, display
from pathlib import Path

html_content = Path(result["html_path"]).read_text()
display(HTML(html_content))

## 7. Multi-Document Example (with relationships)

In [ ]:
# Create two related documents
Path("api_spec.txt").write_text("""
# API Specification
POST /auth/login - Authenticate user, return JWT tokens
POST /auth/refresh - Refresh access token using refresh token
GET /users/me - Get current user profile (requires auth)
""")

Path("security_policy.txt").write_text("""
# Security Policy
All API endpoints must validate JWT signatures.
Tokens must be transmitted over HTTPS only.
Failed auth attempts must be logged and rate-limited.
""")

# Map with relationships
result2 = cognilink.map(
    ["api_spec.txt", "security_policy.txt"],
    model=model,
    relationships=[
        {"upstream": "NODE_SECURITY_POLICY", "downstream": "NODE_API_SPEC", "type": "constrains"}
    ],
    output_dir="./output_multi"
)

print(f"\n✅ Multi-doc graph: {len(result2['nodes'])} nodes, {len(result2['edges'])} edges")

# Display
html2 = Path(result2["html_path"]).read_text()
display(HTML(html2))

## 8. Download Outputs

In [ ]:
from google.colab import files

# Download the HTML visualization
files.download(result["html_path"])
files.download(result["json_path"])